In [ ]:
# Import required libraries and setup database connections=None, date_start=None, date_end=None, 
import pandas as pd
import numpy as np
from database import get_engine
    
# Initialize database connection
engine = get_engine()
print("✅ Libraries imported and database connection established")

def get_h2h_analysis_efficient(athletes=None, events=None, countries=None, 
                               date_start=None, date_end=None, include_segments=True, 
                               min_matches=1):
    """
    Efficient head-to-head analysis using pandas vectorized operations.
    This approach is 10-100x faster than the loop-based version.
    
    Parameters:
    - athletes: List of athlete names to filter by
    - events: List of event names to filter by  
    - countries: List of countries to filter by
    - date_start: Start date (YYYY-MM-DD)
    - date_end: End date (YYYY-MM-DD)
    - include_segments: Whether to include segment-by-segment H2H stats
    - min_matches: Minimum number of head-to-head matchups to include
    
    Returns:
    - DataFrame with H2H statistics
    """
    
    # Build base query with filters
    base_query = """
    SELECT 
        rr.event_id, rr.athlete_id, rr.prog_id, rr.position,
        pm.swimrank, pm.t1rank, pm.bikerank, pm.t2rank, pm.runrank,
        a.full_name as athlete_name,
        e.event_name, e.event_country, e.event_date
    FROM race_results rr
    JOIN position_metrics pm ON rr.event_id = pm.event_id 
        AND rr.athlete_id = pm.athlete_id 
        AND rr.prog_id = pm.prog_id
    JOIN athlete a ON rr.athlete_id = a.athlete_id
    JOIN events e ON rr.event_id = e.event_id AND rr.prog_id = e.prog_id
    WHERE rr.position IS NOT NULL
    """
    
    # Build conditions for filtering
    conditions = []
    
    if athletes:
        athlete_placeholders = ','.join([f"'{name}'" for name in athletes])
        conditions.append(f"a.full_name IN ({athlete_placeholders})")
    
    if events:
        event_placeholders = ','.join([f"'{event}'" for event in events])
        conditions.append(f"e.event_name IN ({event_placeholders})")
        
    if countries:
        country_placeholders = ','.join([f"'{country}'" for country in countries])
        conditions.append(f"e.event_country IN ({country_placeholders})")
        
    if date_start:
        conditions.append(f"e.event_date >= '{date_start}'")
        
    if date_end:
        conditions.append(f"e.event_date <= '{date_end}'")
    
    # Add conditions to base query
    if conditions:
        final_query = base_query + " AND " + " AND ".join(conditions)
    else:
        final_query = base_query
    
    print(f"🔄 Executing query with {len(conditions)} conditions...")
    
    # Load data
    df = pd.read_sql(final_query, engine)
    
    if len(df) == 0:
        print("❌ No data found for the specified filters")
        return pd.DataFrame()
        
    print(f"✅ Loaded {len(df)} race results for {df['athlete_name'].nunique()} athletes")
    
    # Create all possible athlete pairs within each program using efficient pandas merge
    print("🔄 Creating athlete pairs using efficient pandas operations...")
    df_pairs = (
        df.merge(df, on=["prog_id"], suffixes=("_a", "_b"))
        .query("athlete_id_a < athlete_id_b")  # Keep only unique pairs (A,B) not (B,A)
    )
    
    print(f"✅ Created {len(df_pairs)} athlete pair matchups")
    
    
    # Overall race wins (lower position wins)
    df_pairs["win_a"] = df_pairs["position_a"] < df_pairs["position_b"]
    
    # Segment wins (lower rank wins) - only if including segments
    if include_segments:
        df_pairs["win_swim_a"] = df_pairs["swimrank_a"] < df_pairs["swimrank_b"]
        df_pairs["win_t1_a"] = df_pairs["t1rank_a"] < df_pairs["t1rank_b"]
        df_pairs["win_bike_a"] = df_pairs["bikerank_a"] < df_pairs["bikerank_b"]
        df_pairs["win_t2_a"] = df_pairs["t2rank_a"] < df_pairs["t2rank_b"]
        df_pairs["win_run_a"] = df_pairs["runrank_a"] < df_pairs["runrank_b"]

    
    agg_dict = {
        "prog_id": "count",           # Total matches
        "win_a": "sum"                # Overall wins for athlete A
    }
    
    # Add segment aggregations if requested
    if include_segments:
        agg_dict.update({
            "win_swim_a": "sum",
            "win_t1_a": "sum", 
            "win_bike_a": "sum",
            "win_t2_a": "sum",
            "win_run_a": "sum"
        })
    
    h2h = (
        df_pairs
        .groupby(["athlete_id_a", "athlete_id_b", "athlete_name_a", "athlete_name_b"])
        .agg(agg_dict)
        .reset_index()
    )
    
    # Rename columns for clarity
    h2h = h2h.rename(columns={
        #"prog_id": "total_matches",
        "win_a": "wins_a"
    })
    
    # Calculate wins for athlete B and win percentages
    h2h["wins_b"] = h2h["total_matches"] - h2h["wins_a"]
    h2h["win_pct_a"] = h2h["wins_a"] / h2h["total_matches"]
    h2h["win_pct_b"] = h2h["wins_b"] / h2h["total_matches"]
    
    # Calculate segment statistics if requested
    if include_segments:
        h2h['wins_overall_b'] = h2h['matches'] - h2h['wins_overall']
        h2h['wins_swim_b']    = h2h['matches'] - h2h['wins_swim']
        h2h['wins_t1_b']      = h2h['matches'] - h2h['wins_t1']
        h2h['wins_bike_b']    = h2h['matches'] - h2h['wins_bike']
        h2h['wins_t2_b']      = h2h['matches'] - h2h['wins_t2']
        h2h['wins_run_b']     = h2h['matches'] - h2h['wins_run']
        
        
        
        
        
        segments = ["swim", "t1", "bike", "t2", "run"]
        for segment in segments:
            col_wins = f"win_{segment}_a"
            if col_wins in h2h.columns:
                # Rename the aggregated column for consistency
                h2h = h2h.rename(columns={col_wins: f"wins_{segment}_a"})
                
                # Wins for athlete B in this segment
                h2h[f"wins_{segment}_b"] = h2h["total_matches"] - h2h[f"wins_{segment}_a"]
                
                # Win percentages for both athletes
                h2h[f"win_pct_{segment}_a"] = h2h[f"wins_{segment}_a"] / h2h["total_matches"]
                h2h[f"win_pct_{segment}_b"] = h2h[f"wins_{segment}_b"] / h2h["total_matches"]
    
    # Filter by minimum matches
    if min_matches > 1:
        h2h = h2h[h2h["total_matches"] >= min_matches]
        print(f"🔽 Filtered to {len(h2h)} pairs with at least {min_matches} matches")
    
    # Create the bilateral view (both A vs B and B vs A)
    print("🔄 Creating bilateral view for complete analysis...")
    
    # Create the reverse direction (B vs A)
    h2h_reverse = h2h.copy()
    
    # Swap athlete A and B columns
    cols_to_swap = [
        ("athlete_id_a", "athlete_id_b"),
        ("athlete_name_a", "athlete_name_b"),
        ("wins_a", "wins_b"),
        ("win_pct_a", "win_pct_b")
    ]
    
    for col_a, col_b in cols_to_swap:
        if col_a in h2h_reverse.columns and col_b in h2h_reverse.columns:
            h2h_reverse[col_a], h2h_reverse[col_b] = h2h_reverse[col_b], h2h_reverse[col_a]
    
    # Swap segment columns if they exist
    if include_segments:
        for segment in segments:
            cols_to_swap_seg = [
                (f"wins_{segment}_a", f"wins_{segment}_b"),
                (f"win_pct_{segment}_a", f"win_pct_{segment}_b")
            ]
            for col_a, col_b in cols_to_swap_seg:
                if col_a in h2h_reverse.columns and col_b in h2h_reverse.columns:
                    h2h_reverse[col_a], h2h_reverse[col_b] = h2h_reverse[col_b], h2h_reverse[col_a]
    
    # Combine existing code...

    return h2h 

✅ Libraries imported and database connection established


## 1. H2H Analysis Function

The core function that performs head-to-head analysis with flexible filtering and comprehensive statistics.

In [3]:
# Example usage: Get H2H for specific athletes
h2h_stats = get_h2h_analysis_efficient(
    athletes=["Morgan Pearson", "John Reed", "Chase McQueen"],
    date_start="2024-01-01",
    min_matches=1,
    include_segments=True
)

if h2h_stats is not None:
    print(f"Found {len(h2h_stats)} head-to-head matchups")
else:
    print("No head-to-head data found (function returned None)")

# Display the results
if h2h_stats is not None and len(h2h_stats) > 0:
    # Show overall H2H stats
    print("\n=== OVERALL HEAD-TO-HEAD RESULTS ===")
    for _, row in h2h_stats.iterrows():
        print(f"{row['athlete_name_a']} vs {row['athlete_name_b']}: {row['wins_a']}-{row['wins_b']} ({row['win_pct_a']:.1%})")
    
    # Show segment breakdown for first pair
    if 'wins_swim_a' in h2h_stats.columns:
        print(f"\n=== SEGMENT BREAKDOWN (First Pair) ===")
        first_row = h2h_stats.iloc[0]
        print(f"{first_row['athlete_name_a']} vs {first_row['athlete_name_b']}:")
        segments = ['swim', 't1', 'bike', 't2', 'run']
        for segment in segments:
            wins_col = f'wins_{segment}_a'
            total_col = f'total_matches'
            if wins_col in h2h_stats.columns and total_col in h2h_stats.columns:
                wins = first_row[wins_col]
                total = first_row[total_col]
                if pd.notna(wins) and pd.notna(total) and total > 0:
                    pct = wins / total
                    print(f"  {segment.capitalize()}: {wins}/{total} ({pct:.1%})")
    
    # Create visualization matrix
    print(f"\n=== WIN PERCENTAGE MATRIX ===")
    
    # Get unique athletes
    athletes = sorted(list(set(h2h_stats['athlete_name_a'].tolist() + h2h_stats['athlete_name_b'].tolist())))
    
    # Create matrix
    matrix = pd.DataFrame(index=athletes, columns=athletes, dtype=float)
    
    # Fill matrix
    for _, row in h2h_stats.iterrows():
        matrix.loc[row['athlete_name_a'], row['athlete_name_b']] = row['win_pct_a']
        matrix.loc[row['athlete_name_b'], row['athlete_name_a']] = row['win_pct_b']
    
    # Display matrix
    print(matrix.fillna('-'))
    
    # Store in database for Power BI (optional)
    # h2h_stats.to_sql('h2h_athlete_pairs', engine, if_exists='replace', index=False)
    # print(f"\nStored {len(h2h_stats)} H2H records in database table 'h2h_athlete_pairs'")
else:
    print("No head-to-head data found for the specified criteria.")

🔄 Executing query with 2 conditions...
✅ Loaded 34 race results for 3 athletes
🔄 Creating athlete pairs using efficient pandas operations...
✅ Created 13 athlete pair matchups
🔄 Computing win statistics using vectorized operations...
🔄 Aggregating head-to-head statistics...
🔄 Creating bilateral view for complete analysis...
Found 3 head-to-head matchups

=== OVERALL HEAD-TO-HEAD RESULTS ===
Chase McQueen vs Morgan Pearson: 1-3 (25.0%)
Chase McQueen vs John Reed: 3-2 (60.0%)
Morgan Pearson vs John Reed: 2-2 (50.0%)

=== SEGMENT BREAKDOWN (First Pair) ===
Chase McQueen vs Morgan Pearson:
  Swim: 3/4 (75.0%)
  T1: 1/4 (25.0%)
  Bike: 3/4 (75.0%)
  T2: 4/4 (100.0%)
  Run: 0/4 (0.0%)

=== WIN PERCENTAGE MATRIX ===
               Chase McQueen John Reed Morgan Pearson
Chase McQueen              -       0.6           0.25
John Reed                0.4         -            0.5
Morgan Pearson          0.75       0.5              -


In [ ]:
from datetime import date, datetime

def populate_h2h_summary_table_efficient(athletes=None, events=None, countries=None, 
                                        date_start="2023-01-01", date_end=None, min_matches=2, 
                                        replace_existing=True):
    """
    Populate the h2h_summary table using the efficient pandas-based H2H analysis.
    This version is much faster than the loop-based approach.
    
    Parameters:
    - athletes: List of specific athletes to analyze (None = all athletes)
    - events: List of events to include (None = all events)
    - countries: List of countries to include (None = all countries) 
    - date_start: Start date for analysis (default: recent data from 2023)
    - date_end: End date for analysis (None = present)
    - min_matches: Minimum H2H encounters to include in results
    - replace_existing: Whether to replace existing data or append
    """
    
    print("🚀 Populating H2H Summary Table (Efficient Version)")
    print("=" * 60)
    print(f"📅 Date range: {date_start} to {date_end or 'present'}")
    print(f"🎯 Minimum matches: {min_matches}")
    print(f"👥 Athletes filter: {len(athletes) if athletes else 'All'}")
    
    
    # Clear existing data if replacing
    if replace_existing:
        print("🗑️  Clearing existing H2H data...")
        with engine.connect() as conn:
            conn.execute(text("DELETE FROM h2h_summary"))
            conn.commit()
    
    try:
        # Get H2H data using our efficient function
        h2h_data = get_h2h_analysis_efficient(
            athletes=athletes,
            events=events, 
            countries=countries,
            date_start=date_start,
            date_end=date_end,
            min_matches=min_matches,
            include_segments=True
        )
        
    except KeyboardInterrupt:
        print("❌ Process interrupted by user")
        return 0
    except Exception as e:
        print(f"❌ Error during H2H computation: {e}")
        return 0
    
    if len(h2h_data) == 0:
        print("❌ No H2H data to insert")
        return 0
    
    # Prepare data for database insertion
    print(f"💾 Preparing {len(h2h_data)} H2H records for database insertion...")
    
    # Add metadata columns required by the database schema
    h2h_data['date_range_start'] = pd.to_datetime(date_start) if date_start else None
    h2h_data['date_range_end'] = pd.to_datetime(date_end) if date_end else None
    h2h_data['last_updated'] = date.today()
    h2h_data['min_matches_filter'] = min_matches
    
    # Ensure all segment columns exist (fill missing with None)
    segment_columns = []
    for segment in ['swim', 't1', 'bike', 't2', 'run']:
        segment_columns.extend([
            f'{segment}_wins_a', f'{segment}_wins_b', 
            f'win_pct_{segment}_a', f'{segment}_total'
        ])
    
    # Rename segment win percentage columns to match database schema
    rename_map = {}
    for segment in ['swim', 't1', 'bike', 't2', 'run']:
        old_col = f'win_pct_{segment}_a'
        new_col = f'{segment}_win_pct_a'
        if old_col in h2h_data.columns:
            rename_map[old_col] = new_col
    
    if rename_map:
        h2h_data = h2h_data.rename(columns=rename_map)
    
    # Create segment total columns (use total_matches for each segment)
    for segment in ['swim', 't1', 'bike', 't2', 'run']:
        total_col = f'{segment}_total'
        if total_col not in h2h_data.columns:
            h2h_data[total_col] = h2h_data['total_matches']
    
    # Ensure all required columns exist
    required_columns = [
    'athlete_name_a', 'athlete_name_b', 'total_matches', 'wins_a', 'wins_b',
    'win_pct_a', 'win_pct_b', 'date_range_start', 'date_range_end',
    'last_updated', 'min_matches_filter'
]
    all_columns = required_columns + segment_columns
    
    # Fill missing columns with None
    for col in all_columns:
        if col not in h2h_data.columns:
            h2h_data[col] = None
    
    # Select only the columns we need in the correct order
    h2h_data_final = h2h_data[all_columns]
    
    try:
        # Insert data into database 
        records_inserted = h2h_data_final.to_sql(
            'h2h_summary', 
            engine, 
            if_exists='append', 
            index=False,
            method='multi'
        )
        
        print(f"✅ Successfully inserted {len(h2h_data_final)} H2H records")
        
        # Show summary stats
        unique_athletes = len(set(h2h_data['athlete_a'].tolist() + h2h_data['athlete_b'].tolist()))
        
        print(f"📊 Summary:")
        print(f"   • Total records inserted: {len(h2h_data_final):,}")
        print(f"   • Unique athletes: {unique_athletes}")
        print(f"   • Unique athlete pairs: {len(h2h_data_final) // 2:,}")
        print(f"   • Average matches per pair: {h2h_data['total_matches'].mean():.1f}")
        print(f"   • Max matches for any pair: {h2h_data['total_matches'].max()}")
        print(f"   • Date range: {date_start} to {date_end or 'present'}")
        print(f"   • Minimum matches filter: {min_matches}")
        
        return len(h2h_data_final)
        
    except Exception as e:
        print(f"❌ Error inserting data: {e}")
        return 0


def refresh_h2h_data_efficient(date_start="2024-01-01", min_matches=2):
    """Quick refresh of H2H data using the efficient method"""
    print("🔄 Refreshing H2H data with efficient method...")
    return populate_h2h_summary_table_efficient(
        date_start=date_start,
        min_matches=min_matches, 
        replace_existing=True
    )

print("✅ Efficient H2H population functions ready!")
print("🚀 These functions use pandas vectorized operations for much faster performance")

✅ Efficient H2H population functions ready!
🚀 These functions use pandas vectorized operations for much faster performance
